In [0]:
# "ContextName : chromium.media
# Message Id : NL_HBBTV_TA_INFO
# 호출된 데이터 중
# key=ta_duration 값이 2이상 값의 데이터만 이용"

# "1. 월별 UD별 재생 건수
# |기준 월  |  New UD | UD count | Call count | Duration | 누적 New | 누적 UD |
# | 25 - 01 |     5       |    10        | 100          | 3000     | 10           | 100
# | 25 - 02 |     10     |    20        | 230          | 6000     |  30          | 330
# ~~~~`
# | 25 - 12 |     20     |    100      | 230          | 6000     |  350         | 450
# ~~~~
# | 26 - 03 |     30     |     80      | 230          | 6000     |  450         | 880
# "



In [0]:
%sql 
with filtered as (
  SELECT
      mac_addr,               -- 필수 열  
      to_date(date_ym, 'yyyy-MM') as date_ym, 
      normal_log:ta_duration,
      rank() over(partition by mac_addr order by to_date(date_ym, 'yyyy-MM') asc) as rn
  FROM eic_data_ods.tlamp.normal_log_webos24
  WHERE 1=1
      -- AND X_device_country = ''
      AND context_name = 'chromium.media'
      AND message_id = 'NL_HBBTV_TA_INFO'
      AND to_date(date_ym, 'yyyy-MM') >= '2025-01'
      AND to_date(date_ym, 'yyyy-MM') < '2026-04'
      AND mac_addr is not null 
      AND replace(mac_addr, ' ', '') != ''
), agg_data as (
    select date_ym
        , count(distinct case when rn=1 then mac_addr else NULL end) as new_ud -- 해당 달에 처음 노멀로그를 호출한 디바이스 수량
        , count(distinct mac_addr) as ud_count -- 해당 달에 해당 노멀로그를 호출한 전체 디바이스 수량 (고유 디바이스 수)
        , count(mac_addr) as call_count -- 해당 달에 해당 노멀로그를 호출한 총 숫자 (전체 디바이스 수)
        , sum(case when ta_duration >= 2 then ta_duration else 0 end) as duration -- ta_duration >= 2 값의 ta_duration의 합
    from   filtered
    group by date_ym  
) 

select date_format(date_ym, 'yyyy-MM') as date_ym 
       , new_ud
       , ud_count 
       , call_count   
       , duration 
       , sum(coalesce(new_ud, 0)) over(order by date_ym rows between unbounded PRECEDING and current row) as `누적_new_ud`
       , sum(coalesce(ud_count, 0)) over(order by date_ym rows between unbounded PRECEDING and current row) as `누적_ud_count`
from   agg_data
